# Sparkle V2 — Fase 3.2a: Extracción offline de embeddings CNN (rama de apariencia)

**Entorno de ejecución:** Google Colab (GPU T4) o Kaggle Notebooks (GPU) — **requiere el video crudo de DAiSEE** (a diferencia de los notebooks de Fase 3.0/3.1, que solo usan las features ya extraídas).

**Objetivo (Roadmap Fase 3.2, semana 1-2 del plan):** implementar la palanca principal que identifica el diagnóstico del plan (§0, §2): la señal geométrica (landmarks/blendshapes) sola no alcanza ~50% en DAiSEE; los trabajos que superan 65-73% usan apariencia/textura vía CNN sobre el frame o el crop facial. Este notebook agrega ese segundo extractor **sin romper el contrato de privacidad ya vigente**: el crop facial se procesa en memoria, se reduce a un embedding de baja dimensión, y se descarta inmediatamente — exactamente igual a como hoy se descartan los frames tras calcular blendshapes.

**Este notebook corre UNA SOLA VEZ por dataset** (extracción offline, §2.4 del plan): produce embeddings persistidos en `.npz`, igual que ya hacen con las features geométricas. El entrenamiento del modelo temporal (notebook `Fase3_2b`) nunca vuelve a tocar el video.

**Estimación de costo:** 8.416 clips × 16 frames ≈ 135k inferencias de CNN ≈ 1-2 h en GPU T4 (bastante menos que la extracción de landmarks de Fase 2, que ya corrieron sobre los 60 frames completos).

## 0. Decisión de diseño: backbone y pesos (§2.2 del plan)

El plan es explícito: **"no usar pesos ImageNet a secas"** — un backbone preentrenado en rostros/afecto rinde mucho más con pocos datos. Este notebook implementa la **Opción A** (recomendada): fine-tuning previo de MobileNetV2 en un dataset de expresión facial abierto, usando **FER2013** (más simple de conseguir en Kaggle que AffectNet/RAF-DB, que requieren aprobación de acceso) como tarea proxy de emociones.

Si el fine-tuning en FER2013 no está disponible o se quiere iterar más rápido primero, la celda de la sección 2 tiene un flag `USE_FER_FINETUNED = False` para caer a la **Opción B** (pesos ImageNet directos) — el plan advierte que esto rinde peor, así que se recomienda tratarlo solo como punto de partida para validar que el pipeline corre de punta a punta, no como configuración final.

In [ ]:
# Colab: Runtime -> Change runtime type -> GPU (T4)
!pip install -q mediapipe tensorflow scikit-learn

import os, glob, math, time
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from tqdm.notebook import tqdm

print("TF:", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))

# from google.colab import drive
# drive.mount('/content/drive')

PROJECT_DIR   = '/content/drive/MyDrive/SparkleV2'
RAW_DATASET_DIR = '/content/daisee_raw/DAiSEE/DataSet'
LABELS_DIR    = f'{PROJECT_DIR}/labels'
EMB_DIR       = f'{PROJECT_DIR}/embeddings_cnn'          # NUEVO: embeddings, no features geometricas
BACKBONE_DIR  = f'{PROJECT_DIR}/checkpoints_fase3_2'
for split in ['Train', 'Validation', 'Test']:
    os.makedirs(f'{EMB_DIR}/{split}', exist_ok=True)
os.makedirs(BACKBONE_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

CROP_SIZE = 112          # MobileNetV2 alpha=0.35 @112 (§2.2 del plan)
N_EMB_FRAMES = 16        # muestreo temporal disperso (§2.3): 0.8-1.6 fps sobre 10s
EMB_DIM_PROJ = 128       # proyeccion final del embedding (64-128d, §2.1)
ALPHA = 0.35

In [ ]:
if not os.path.exists(RAW_DATASET_DIR):
    print('Descargando DAiSEE (video crudo) desde Kaggle...')
    if not os.path.exists('/root/.kaggle/kaggle.json'):
        from google.colab import files
        print('Subi tu kaggle.json:')
        uploaded = files.upload()
        os.makedirs('/root/.kaggle', exist_ok=True)
        !cp kaggle.json /root/.kaggle/kaggle.json
        !chmod 600 /root/.kaggle/kaggle.json
    os.makedirs('/content/daisee_raw', exist_ok=True)
    !kaggle datasets download -d olgaparfenova/daisee -p /content/daisee_raw --unzip
else:
    print('Dataset ya presente en este runtime.')

def load_labels(split):
    df = pd.read_csv(f'{LABELS_DIR}/{split}Labels.csv')
    df.columns = [c.strip() for c in df.columns]
    return df

labels = {s: load_labels(s) for s in ['Train', 'Validation', 'Test']}
video_index = {}
for split in ['Train', 'Validation', 'Test']:
    for p in glob.glob(f'{RAW_DATASET_DIR}/{split}/**/*.avi', recursive=True):
        video_index[os.path.basename(p)] = p
print(f'Videos indexados: {len(video_index)}')

## 1. FaceLandmarker (mismo modelo que Fase 2/3) para obtener el bounding box del rostro

Se reutiliza el `FaceLandmarker` de MediaPipe ya validado en la extracción de blendshapes — evita introducir un segundo detector facial. El bounding box se deriva de los landmarks (min/max x,y con margen), no hace falta un detector de caras separado.

In [ ]:
MODEL_PATH = '/content/face_landmarker.task'
if not os.path.exists(MODEL_PATH):
    !wget -q -O {MODEL_PATH} https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task

def create_face_landmarker():
    base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
    options = mp_vision.FaceLandmarkerOptions(
        base_options=base_options, running_mode=mp_vision.RunningMode.IMAGE,
        num_faces=1, min_face_detection_confidence=0.5)
    return mp_vision.FaceLandmarker.create_from_options(options)

def crop_face_from_landmarks(frame_bgr, landmarks, crop_size=CROP_SIZE, margin=0.25):
    """Bounding box a partir de min/max de todos los landmarks + margen relativo,
    recorte cuadrado (evita distorsionar el rostro) y resize a crop_size x crop_size.
    Devuelve RGB en [0,1] listo para MobileNetV2 (preprocess_input se aplica aparte)."""
    h, w = frame_bgr.shape[:2]
    xs = np.array([lm.x for lm in landmarks]) * w
    ys = np.array([lm.y for lm in landmarks]) * h
    x_min, x_max, y_min, y_max = xs.min(), xs.max(), ys.min(), ys.max()
    bw, bh = x_max - x_min, y_max - y_min
    side = max(bw, bh) * (1 + margin)
    cx, cy = (x_min + x_max) / 2, (y_min + y_max) / 2
    x0, y0 = int(max(0, cx - side / 2)), int(max(0, cy - side / 2))
    x1, y1 = int(min(w, cx + side / 2)), int(min(h, cy + side / 2))
    if x1 <= x0 or y1 <= y0:
        return None
    crop = frame_bgr[y0:y1, x0:x1]
    crop = cv2.resize(crop, (crop_size, crop_size), interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

def extract_crop_from_frame(detector, frame_bgr):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    result = detector.detect(mp_image)
    if not result.face_landmarks:
        return None
    return crop_face_from_landmarks(frame_bgr, result.face_landmarks[0])

## 2. Backbone CNN — MobileNetV2 α=0,35 @112 (fine-tuned en FER2013 o ImageNet directo)

**Opción A (recomendada, `USE_FER_FINETUNED = True`):** fine-tuning corto de MobileNetV2 en FER2013 (7 clases de emoción, imágenes 48×48 en escala de grises — se convierten a RGB 112×112 con resize). Al terminar, se congela el backbone y se usa la penúltima capa (antes del clasificador de 7 clases) como extractor de "textura afectiva".

**Opción B (`USE_FER_FINETUNED = False`):** MobileNetV2 con pesos ImageNet directos, sin fine-tuning. Más rápido de poner en marcha pero el plan advierte que rinde peor — usar solo para validar el pipeline end-to-end antes de invertir en el fine-tuning.

In [ ]:
USE_FER_FINETUNED = True

def build_backbone(alpha=ALPHA, crop_size=CROP_SIZE, weights='imagenet'):
    return keras.applications.MobileNetV2(
        input_shape=(crop_size, crop_size, 3), alpha=alpha,
        include_top=False, weights=weights, pooling='avg')

if USE_FER_FINETUNED:
    # --- Descargar FER2013 (Kaggle) ---
    fer_dir = '/content/fer2013'
    if not os.path.exists(fer_dir):
        os.makedirs(fer_dir, exist_ok=True)
        # Verificar el slug del dataset antes de correr: hay varios mirrors de FER2013 en Kaggle.
        !kaggle datasets download -d msambare/fer2013 -p {fer_dir} --unzip

    fer_train_dir = f'{fer_dir}/train'
    img_gen = keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=keras.applications.mobilenet_v2.preprocess_input,
        validation_split=0.1)
    train_gen = img_gen.flow_from_directory(
        fer_train_dir, target_size=(CROP_SIZE, CROP_SIZE), color_mode='rgb',
        batch_size=64, class_mode='categorical', subset='training', seed=SEED)
    val_gen = img_gen.flow_from_directory(
        fer_train_dir, target_size=(CROP_SIZE, CROP_SIZE), color_mode='rgb',
        batch_size=64, class_mode='categorical', subset='validation', seed=SEED)
    n_fer_classes = train_gen.num_classes
    print(f'FER2013: {train_gen.samples} train, {val_gen.samples} val, {n_fer_classes} clases')

    backbone = build_backbone(weights='imagenet')
    backbone.trainable = True   # fine-tuning completo (backbone es chico, alpha=0.35)
    fer_head = layers.Dense(n_fer_classes, activation='softmax', name='fer_head')(backbone.output)
    fer_model = keras.Model(backbone.input, fer_head)
    fer_model.compile(optimizer=keras.optimizers.Adam(1e-4),
                       loss='categorical_crossentropy', metrics=['accuracy'])
    fer_model.fit(train_gen, validation_data=val_gen, epochs=10,
                  callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)])
    backbone.trainable = False
    backbone.save(f'{BACKBONE_DIR}/mobilenetv2_fer_finetuned.keras')
    print('Backbone fine-tuned en FER2013 y congelado.')
else:
    backbone = build_backbone(weights='imagenet')
    backbone.trainable = False
    print('Backbone con pesos ImageNet directos (Opcion B, rendimiento esperado menor).')

# Extractor final: backbone congelado + proyeccion a EMB_DIM_PROJ
inp_crop = keras.Input(shape=(CROP_SIZE, CROP_SIZE, 3), name='crop')
feat = backbone(inp_crop, training=False)
emb = layers.Dense(EMB_DIM_PROJ, activation=None, name='embedding_projection')(feat)
extractor_model = keras.Model(inp_crop, emb, name='cnn_extractor')
print(f'Extractor listo: {extractor_model.count_params():,} parametros, salida {extractor_model.output_shape}')

## 3. Función de procesamiento por clip — muestreo disperso (16 frames) + embedding

A diferencia de la extracción geométrica (60 frames/clip), acá se muestrean solo 16 frames por clip (§2.3 del plan: 0,8-1,6 fps es suficiente porque la textura facial cambia lento). El frame se descarta inmediatamente después de convertirlo en embedding — nunca se persiste el crop ni el frame.

In [ ]:
def process_clip_embeddings(video_path, detector, extractor, n_frames=N_EMB_FRAMES):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release(); return None

    frame_indices = np.linspace(0, total_frames - 1, n_frames).astype(int)
    crops = np.zeros((n_frames, CROP_SIZE, CROP_SIZE, 3), dtype=np.float32)
    valid_mask = np.zeros(n_frames, dtype=bool)

    for seq_pos, frame_idx in enumerate(frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ret, frame = cap.read()
        if not ret:
            continue
        crop = extract_crop_from_frame(detector, frame)   # frame se descarta al salir del scope
        if crop is not None:
            crops[seq_pos] = keras.applications.mobilenet_v2.preprocess_input(crop.astype(np.float32))
            valid_mask[seq_pos] = True
    cap.release()

    embeddings = extractor.predict(crops, batch_size=n_frames, verbose=0)   # (16, EMB_DIM_PROJ)
    embeddings[~valid_mask] = np.nan   # frames sin rostro detectado -> NaN (mismo tratamiento que geometricas)
    return embeddings

# --- Prueba en un subset chico (igual patron que la extraccion geometrica de Fase 2) ---
test_clips = labels['Train'].head(10)
detector = create_face_landmarker()
try:
    t0 = time.time()
    for _, row in test_clips.iterrows():
        clip_id = row['ClipID']
        if clip_id not in video_index:
            continue
        emb = process_clip_embeddings(video_index[clip_id], detector, extractor_model)
        pct_nan = np.isnan(emb).any(axis=1).mean() * 100
        print(f'{clip_id}: {pct_nan:.1f}% frames sin rostro | embedding shape {emb.shape} '
              f'| norma media: {np.nanmean(np.linalg.norm(emb, axis=1)):.3f}')
    elapsed = time.time() - t0
finally:
    detector.close()
print(f'\nTiempo: {elapsed:.1f}s para 10 clips ({elapsed/10:.2f}s/clip)')
print(f'Estimado dataset completo (~7900 clips): {elapsed/10*7900/60:.1f} minutos')

**Antes de lanzar el batch completo:** confirmar que el % de frames sin rostro es razonable (similar al de la extracción geométrica) y que la norma de los embeddings no es 0 ni constante entre clips distintos.

In [ ]:
SPLIT = 'Train'  # cambiar a 'Validation' o 'Test' cuando corresponda

df_split = labels[SPLIT]
out_dir = f'{EMB_DIR}/{SPLIT}'
log_path = f'{EMB_DIR}/{SPLIT}_processing_log.csv'
log_rows = pd.read_csv(log_path).to_dict('records') if os.path.exists(log_path) else []
already_done = set(os.path.splitext(f)[0] for f in os.listdir(out_dir) if f.endswith('.npz'))
print(f'{SPLIT}: {len(already_done)} clips ya procesados de {len(df_split)} totales')

detector = create_face_landmarker()
try:
    for _, row in tqdm(df_split.iterrows(), total=len(df_split)):
        clip_id = row['ClipID']
        clip_key = os.path.splitext(clip_id)[0]
        if clip_key in already_done:
            continue
        if clip_id not in video_index:
            log_rows.append({'ClipID': clip_id, 'status': 'sin_video_fisico'}); continue
        try:
            emb = process_clip_embeddings(video_index[clip_id], detector, extractor_model)
            if emb is None:
                log_rows.append({'ClipID': clip_id, 'status': 'video_ilegible'}); continue
            pct_nan = np.isnan(emb).any(axis=1).mean() * 100
            np.savez_compressed(f'{out_dir}/{clip_key}.npz', embeddings=emb.astype(np.float32))
            log_rows.append({'ClipID': clip_id, 'status': 'ok', 'pct_nan': pct_nan})
        except Exception as e:
            log_rows.append({'ClipID': clip_id, 'status': f'error: {e}'})
        if len(log_rows) % 50 == 0:
            pd.DataFrame(log_rows).to_csv(log_path, index=False)
finally:
    detector.close()

pd.DataFrame(log_rows).to_csv(log_path, index=False)
print('Terminado (o cortado). Progreso guardado en', log_path)
print('Repetir esta celda cambiando SPLIT para Validation y Test.')

## Checkpoint de la Fase 3.2a

1. Correr la sección 3 (subset de 10 clips) y confirmar que los embeddings tienen norma no-trivial y variabilidad entre clips (si todos dan casi el mismo vector, algo está mal en el crop o el backbone).
2. Repetir la celda de batch completo para `Train`, `Validation` y `Test`.
3. **No se persiste ningún crop ni frame** — solo el vector de embedding de 128-d por frame muestreado, igual contrato de privacidad que ya usan (§2.5 del plan: nunca sale del dispositivo, se descarta tras la inferencia; acá aplica lo mismo a nivel de extracción offline del dataset de entrenamiento).
4. Con los tres splits listos, seguir con `Sparkle_V2_Fase3_2b_Fusion_Apariencia_Geometria.ipynb` para entrenar el modelo de fusión.